# 02. Вимір валют

Цей ноутбук відповідає за другий етап конвеєра. Він читає сирі дані з bronze-таблиці, розбирає JSON та створює довідник валют.

У таблиці `dim_currency` кожна валюта повинна зустрічатися лише один раз. Для кожної валюти зберігатимуться її код, назва, цифровий код та окремий ключ.

Цей ноутбук не звертається до API НБУ. Він читає дані тільки з таблиці `nbu_raw.raw_rates`, яку підготував попередній ноутбук.

Результат записується в режимі `WRITE_TRUNCATE`. Це означає, що під час кожного запуску таблиця виміру будується заново з усіх наявних даних bronze.

## Послідовність роботи

1. Підключитися до BigQuery.
2. Прочитати потрібні колонки з bronze-таблиці.
3. Розгорнути JSON із колонки `payload`.
4. Очистити та привести значення до потрібних типів.
5. Залишити один найсвіжіший рядок для кожної валюти.
6. Додати послідовний ключ `currency_key`.
7. Додати технічний рядок `Unknown`.
8. Записати результат у `nbu_dwh.dim_currency`.
9. Перевірити отриману таблицю.

## 1. Налаштування та підключення

Імпортуємо бібліотеки, вказуємо назву Google Cloud проєкту та створюємо клієнт BigQuery.

Кожен ноутбук підключається до BigQuery самостійно. Це потрібно тому, що надалі оркестратор запускатиме ноутбуки як окремі процеси.

Для авторизації використовуємо JSON-ключ сервісного акаунта зі змінної середовища `GCP_SA_KEY`. Сам ключ у коді не зберігається.

In [1]:
PROJECT_ID = "nbu-bigquery-etl"
LOCATION = "EU"

DS_RAW = "nbu_raw"
DS_DWH = "nbu_dwh"

RAW_TABLE = f"{PROJECT_ID}.{DS_RAW}.raw_rates"
DIM_CURRENCY_TABLE = f"{PROJECT_ID}.{DS_DWH}.dim_currency"

In [2]:
import os
import sys
import json

import pandas as pd
from google.cloud import bigquery

pd.set_option("display.max_columns", 40)

In [3]:
creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account

    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"]

    )

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)

print("проєкт:", client.project)

проєкт: nbu-bigquery-etl


## Завдання 3. Ноутбук 02: вимір валют

### Завдання 3.1. Читання даних із bronze

Читаємо з таблиці `nbu_raw.raw_rates` тільки три колонки: момент завантаження, дату курсу та JSON-текст із даними валюти.

Момент завантаження знадобиться пізніше, щоб серед повторних записів вибрати найсвіжіший. Колонка `payload` містить інформацію про валюту, яку на наступному кроці ми розгорнемо в окремі колонки.

На цьому етапі ми нічого не змінюємо і не видаляємо. Спочатку лише отримуємо дані з попереднього шару конвеєра.

In [4]:
sql = f"""
SELECT ingested_at,
       business_date,
       payload
  FROM `{RAW_TABLE}`
"""

raw = client.query(sql).to_dataframe()

print(len(raw), "рядків прочитано з bronze")
raw.head(5)

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


90 рядків прочитано з bronze


,ingested_at,business_date,payload
0,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""DZD"", ""exchangedate"": ""24.08.2026"", ""r..."
1,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""AUD"", ""exchangedate"": ""24.08.2026"", ""r..."
2,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""BDT"", ""exchangedate"": ""24.08.2026"", ""r..."
3,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""CAD"", ""exchangedate"": ""24.08.2026"", ""r..."
4,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""CNY"", ""exchangedate"": ""24.08.2026"", ""r..."


### Завдання 3.2. Розгортання JSON у колонки

У bronze інформація про кожну валюту зберігається в колонці `payload` як JSON-текст. У такому вигляді дані зручно зберігати, але незручно очищувати та аналізувати.

На цьому етапі перетворюємо JSON-текст на окремі колонки. Після перетворення отримаємо поля `r030`, `txt`, `rate`, `cc` та `exchangedate`.

На цьому етапі працюємо тільки з колонкою `payload`. Службові колонки залишаються в DataFrame `raw`. Після очищення JSON-полів ми приєднаємо `ingested_at`, оскільки він потрібен для вибору найсвіжішого запису валюти.

Колонку `business_date` у вимір валют не переносимо, тому що цей вимір описує валюту, а не її курс за певну дату.

In [5]:
parsed = pd.json_normalize(raw["payload"].map(json.loads))

print("колонки JSON:", list(parsed.columns))
print(len(parsed), "рядків після розгортання JSON")

parsed.head(5)

колонки JSON: ['cc', 'exchangedate', 'r030', 'rate', 'special', 'txt']
90 рядків після розгортання JSON


,cc,exchangedate,r030,rate,special,txt
0,DZD,24.08.2026,12,0.33613,NaN,Алжирський динар
1,AUD,24.08.2026,36,31.99060,NaN,Австралійський долар
2,BDT,24.08.2026,50,0.36509,NaN,Така
3,CAD,24.08.2026,124,32.50720,NaN,Канадський долар
4,CNY,24.08.2026,156,6.64470,NaN,Юань Женьміньбі


### Завдання 3.3. Нормалізація значень

Очищуємо значення, отримані з JSON. Код валюти приводимо до верхнього регістру, назву очищуємо від зайвих пробілів, а цифровий код перетворюємо на цілий тип `Int64`.

Після очищення залишаємо тільки ті JSON-поля, які потрібні для побудови виміру валют. До них приєднуємо колонку `ingested_at` із bronze. Вона потрібна, щоб серед повторних записів вибрати найсвіжішу версію кожної валюти.

Колонку `business_date` не переносимо, оскільки вона не використовується для побудови `dim_currency`.

In [6]:
parsed["currency_code"] = parsed["cc"].str.strip().str.upper()
parsed["currency_name"] = parsed["txt"].str.strip()
parsed["r030"] = pd.to_numeric(parsed["r030"], errors='coerce').astype('Int64')

currency_data = parsed[["currency_code", "currency_name", "r030"]].copy()

df = pd.concat(
    [
        raw[["ingested_at"]].reset_index(drop=True),
        currency_data.reset_index(drop=True)
    ],
    axis=1
)

print("колонки:", list(df.columns))
print(len(df), "рядків після нормалізації")

df.head(5)

колонки: ['ingested_at', 'currency_code', 'currency_name', 'r030']
90 рядків після нормалізації


,ingested_at,currency_code,currency_name,r030
0,2026-08-24 08:14:35+00:00,DZD,Алжирський динар,12
1,2026-08-24 08:14:35+00:00,AUD,Австралійський долар,36
2,2026-08-24 08:14:35+00:00,BDT,Така,50
3,2026-08-24 08:14:35+00:00,CAD,Канадський долар,124
4,2026-08-24 08:14:35+00:00,CNY,Юань Женьміньбі,156


### Завдання 3.4. Вибір найсвіжішого запису валюти

У bronze одна валюта може повторюватися після кількох запусків завантаження. У вимірі кожна валюта повинна бути представлена лише одним рядком.

Спочатку сортуємо записи за `ingested_at` від найстаріших до найновіших. Потім для кожного `currency_code` залишаємо останній рядок. Таким способом отримуємо найсвіжішу версію назви та цифрового коду валюти.

Після дедуплікації колонка `ingested_at` більше не потрібна, тому не включаємо її до результату.

In [7]:
dim = df.sort_values(["ingested_at"]).drop_duplicates(subset=["currency_code"],
                                      keep="last")[["currency_code", "currency_name", "r030"]].copy()

print(len(df), "рядків до дедуплікації")
print(len(dim), "унікальних валют")
dim.head(5)

90 рядків до дедуплікації
45 унікальних валют


,currency_code,currency_name,r030
75,EGP,Єгипетський фунт,818
74,TND,Туніський динар,788
73,AED,Дирхам ОАЕ,784
69,ZAR,Ренд,710
71,CHF,Швейцарський франк,756


### Завдання 3.5. Створення сурогатного ключа

Сортуємо валюти за `currency_code`, щоб порядок був передбачуваним під час повторного запуску. Після сортування оновлюємо індекс і додаємо першою колонкою послідовний ключ `currency_key` від 1.

`currency_key` є внутрішнім ключем сховища. Він не приходить від НБУ, а створюється під час побудови виміру.

In [8]:
dim = dim.sort_values(["currency_code"]).reset_index(drop=True)

dim.insert(0, "currency_key", range(1, len(dim) + 1))

dim.head(5)

,currency_key,currency_code,currency_name,r030
0,1,AED,Дирхам ОАЕ,784
1,2,AUD,Австралійський долар,36
2,3,AZN,Азербайджанський манат,944
3,4,BDT,Така,50
4,5,CAD,Канадський долар,124


### Завдання 3.6. Додавання рядка Unknown

Додаємо технічний рядок `Unknown` із ключем `-1`. Він використовуватиметься тоді, коли код валюти з таблиці фактів не буде знайдений у вимірі.

Такий рядок дозволяє не залишати зовнішній ключ порожнім і не втрачати фактичний запис через відсутність відповідної валюти в довіднику.

In [9]:
dim.loc[len(dim)] = {"currency_key": -1, "currency_code": "N/A", "currency_name": "Unknown", "r030": -1}
dim.tail(5)

,currency_key,currency_code,currency_name,r030
41,42,XDR,СПЗ (спеціальні права запозичення),960
42,43,XPD,Паладій,964
43,44,XPT,Платина,962
44,45,ZAR,Ренд,710
45,-1,N/A,Unknown,-1


### Завдання 3.7. Запис виміру валют у BigQuery

Записуємо підготовлений DataFrame у таблицю `nbu_dwh.dim_currency`.

Використовуємо режим `WRITE_TRUNCATE`. Під час першого запуску BigQuery створить таблицю, а під час наступних запусків повністю замінить її вміст новим результатом.

Схему таблиці описуємо явно. Це дозволяє заздалегідь визначити назви колонок, їхні типи та заборонити порожні значення через `mode="REQUIRED"`.

Схему задаємо вручну, щоб BigQuery не визначав типи колонок самостійно. Якщо назви або типи колонок не збігатимуться, ми одразу побачимо помилку під час завантаження.

Такий режим підходить для виміру валют, оскільки таблиця повністю будується з даних bronze. Повторний запуск із тими самими вхідними даними повинен дати такий самий результат без додавання дублікатів.

In [10]:
dim_currency_schema = [
    bigquery.SchemaField("currency_key",   "INTEGER",   mode="REQUIRED"),
    bigquery.SchemaField("currency_code",  "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("currency_name",  "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("r030",           "INTEGER",   mode="REQUIRED"),
]

cfg = bigquery.LoadJobConfig(schema=dim_currency_schema, write_disposition="WRITE_TRUNCATE")

load_job = client.load_table_from_dataframe(dim, DIM_CURRENCY_TABLE, job_config=cfg)

load_job.result()

table = client.get_table(DIM_CURRENCY_TABLE)

print("таблиця:", DIM_CURRENCY_TABLE)
print("записано рядків:", table.num_rows)

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


таблиця: nbu-bigquery-etl.nbu_dwh.dim_currency
записано рядків: 46


### Завдання 3.8. Перевірка виміру валют

Читаємо записану таблицю `nbu_dwh.dim_currency` із BigQuery та перевіряємо три умови.

Код кожної валюти повинен бути унікальним. У таблиці має бути технічний рядок із ключем `-1`. Загальна кількість рядків повинна дорівнювати кількості унікальних валют у bronze плюс один рядок `Unknown`.

Ці перевірки підтверджують, що таблиця має правильне зерно, містить технічний рядок і не втратила валюти під час перетворення.

In [11]:
sql = f"""
SELECT currency_key,
       currency_code,
       currency_name,
       r030
  FROM `{DIM_CURRENCY_TABLE}`
 ORDER BY currency_key
"""

check_dim = client.query(sql).to_dataframe()

print(len(check_dim), f"рядків прочитано з {DIM_CURRENCY_TABLE}")
check_dim.head(5)

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


46 рядків прочитано з nbu-bigquery-etl.nbu_dwh.dim_currency


,currency_key,currency_code,currency_name,r030
0,-1,N/A,Unknown,-1
1,1,AED,Дирхам ОАЕ,784
2,2,AUD,Австралійський долар,36
3,3,AZN,Азербайджанський манат,944
4,4,BDT,Така,50


In [12]:
print("currency_code унікальний:", check_dim["currency_code"].is_unique)

print("рядок із currency_key = -1 існує:", (check_dim["currency_key"] == -1).any())

print("кількість рядків правильна:", len(check_dim) == currency_data["currency_code"].nunique() + 1)

currency_code унікальний: True
рядок із currency_key = -1 існує: True
кількість рядків правильна: True


In [13]:
print("02_dim_currency завершено успішно")

02_dim_currency завершено успішно
